<a href="https://colab.research.google.com/github/AnnaZapototska/goit-rdb-hw-03/blob/development/hw_db_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

In [2]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=222)

df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Shape: (45211, 17)

Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

First 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [4]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nUnique values in categorical columns:")
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")


Data types:
age             int64
job            object
marital        object
education      object
default        object
balance         int64
housing        object
loan           object
contact        object
day_of_week     int64
month          object
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome       object
y              object
dtype: object

Missing values:
poutcome       36959
contact        13020
education       1857
job              288
age                0
default            0
balance            0
housing            0
marital            0
loan               0
day_of_week        0
duration           0
month              0
campaign           0
pdays              0
previous           0
y                  0
dtype: int64

Unique values in categorical columns:
job: 11 unique values
marital: 3 unique values
education: 3 unique values
default: 2 unique values
housing: 2 unique values
loan: 2 unique values
contact: 2 unique values
mon

In [5]:
import os

csv_path = "/content/hw3_bank_marketing.csv"

df.to_csv(
    csv_path,
    index=False,
    na_rep="",
    encoding="utf-8"
)

print("CSV saved:", os.path.exists(csv_path))
print("File size (MB):", round(os.path.getsize(csv_path) / (1024 ** 2), 2))
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(pd.read_csv(csv_path, nrows=5))


CSV saved: True
File size (MB): 3.19
Rows: 45211
Columns: 17


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [6]:
create_staging_sql = """
DROP TABLE IF EXISTS hw3_bank_clean CASCADE;
DROP TABLE IF EXISTS hw3_bank_staging CASCADE;

CREATE TABLE hw3_bank_staging (
    age_raw          TEXT,
    job_raw          TEXT,
    marital_raw      TEXT,
    education_raw    TEXT,
    default_raw      TEXT,
    balance_raw      TEXT,
    housing_raw      TEXT,
    loan_raw         TEXT,
    contact_raw      TEXT,
    day_of_week_raw  TEXT,
    month_raw        TEXT,
    duration_raw     TEXT,
    campaign_raw     TEXT,
    pdays_raw        TEXT,
    previous_raw     TEXT,
    poutcome_raw     TEXT,
    y_raw            TEXT
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_staging_sql)

print("Staging table created.")


Staging table created.


In [7]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(
                """
                COPY hw3_bank_staging (
                    age_raw,
                    job_raw,
                    marital_raw,
                    education_raw,
                    default_raw,
                    balance_raw,
                    housing_raw,
                    loan_raw,
                    contact_raw,
                    day_of_week_raw,
                    month_raw,
                    duration_raw,
                    campaign_raw,
                    pdays_raw,
                    previous_raw,
                    poutcome_raw,
                    y_raw
                )
                FROM STDIN
                WITH (
                    FORMAT CSV,
                    HEADER TRUE,
                    DELIMITER ',',
                    NULL ''
                )
                """,
                file
            )

    raw_conn.commit()
finally:
    raw_conn.close()

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM hw3_bank_staging")
    ).scalar_one()

print("Rows loaded:", result)


Rows loaded: 45211


In [8]:
with engine.connect() as conn:
    print("Row count:")
    print(pd.read_sql(
        "SELECT COUNT(*) AS n_rows FROM hw3_bank_staging",
        conn
    ))

    print("\nFirst 5 rows:")
    display(pd.read_sql(
        "SELECT * FROM hw3_bank_staging LIMIT 5",
        conn
    ))

    print("\nRaw NULL counts:")
    null_counts = conn.execute(text("""
        SELECT
            COUNT(*) FILTER (WHERE job_raw IS NULL) AS job_nulls,
            COUNT(*) FILTER (WHERE education_raw IS NULL) AS education_nulls,
            COUNT(*) FILTER (WHERE contact_raw IS NULL) AS contact_nulls,
            COUNT(*) FILTER (WHERE poutcome_raw IS NULL) AS poutcome_nulls
        FROM hw3_bank_staging
    """)).mappings().one()

    print(dict(null_counts))

Row count:
   n_rows
0   45211

First 5 rows:


,age_raw,job_raw,marital_raw,education_raw,default_raw,balance_raw,housing_raw,loan_raw,contact_raw,day_of_week_raw,month_raw,duration_raw,campaign_raw,pdays_raw,previous_raw,poutcome_raw,y_raw
0,58,management,married,tertiary,no,2143,yes,no,None,5,may,261,1,-1,0,None,no
1,44,technician,single,secondary,no,29,yes,no,None,5,may,151,1,-1,0,None,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,None,5,may,76,1,-1,0,None,no
3,47,blue-collar,married,None,no,1506,yes,no,None,5,may,92,1,-1,0,None,no
4,33,None,single,None,no,1,no,no,None,5,may,198,1,-1,0,None,no



Raw NULL counts:
{'job_nulls': 288, 'education_nulls': 1857, 'contact_nulls': 13020, 'poutcome_nulls': 36959}


In [9]:
with engine.connect() as conn:
    for col in ["default_raw", "housing_raw", "loan_raw", "y_raw"]:
        print(f"\n{col}:")
        display(pd.read_sql(
            text(f"""
                SELECT {col}, COUNT(*) AS n_rows
                FROM hw3_bank_staging
                GROUP BY {col}
                ORDER BY n_rows DESC
            """),
            conn
        ))

    print("\nDistinct month values:")
    display(pd.read_sql(
        """
        SELECT month_raw, COUNT(*) AS n_rows
        FROM hw3_bank_staging
        GROUP BY month_raw
        ORDER BY month_raw
        """,
        conn
    ))


default_raw:


,default_raw,n_rows
0,no,44396
1,yes,815



housing_raw:


,housing_raw,n_rows
0,yes,25130
1,no,20081



loan_raw:


,loan_raw,n_rows
0,no,37967
1,yes,7244



y_raw:


,y_raw,n_rows
0,no,39922
1,yes,5289



Distinct month values:


,month_raw,n_rows
0,apr,2932
1,aug,6247
2,dec,214
3,feb,2649
4,jan,1403
5,jul,6895
6,jun,5341
7,mar,477
8,may,13766
9,nov,3970


In [10]:
create_clean_sql = """
DROP TABLE IF EXISTS hw3_bank_clean CASCADE;

CREATE TABLE hw3_bank_clean (
    record_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    age SMALLINT NOT NULL
        CHECK (age > 0),

    job TEXT,
    marital TEXT,
    education TEXT,

    default_credit BOOLEAN NOT NULL,
    balance INTEGER NOT NULL,

    housing BOOLEAN NOT NULL,
    loan BOOLEAN NOT NULL,

    contact TEXT,

    day_of_week SMALLINT NOT NULL
        CHECK (day_of_week BETWEEN 1 AND 31),

    month TEXT NOT NULL
        CHECK (
            month IN (
                'jan', 'feb', 'mar', 'apr', 'may', 'jun',
                'jul', 'aug', 'sep', 'oct', 'nov', 'dec'
            )
        ),

    duration INTEGER NOT NULL
        CHECK (duration >= 0),

    campaign SMALLINT NOT NULL
        CHECK (campaign > 0),

    pdays INTEGER NOT NULL
        CHECK (pdays = -1 OR pdays >= 0),

    previous SMALLINT NOT NULL
        CHECK (previous >= 0),

    poutcome TEXT,

    y BOOLEAN NOT NULL
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_clean_sql)

print("Table hw3_bank_clean created.")

Table hw3_bank_clean created.


In [11]:
insert_clean_sql = """
INSERT INTO hw3_bank_clean (
    age,
    job,
    marital,
    education,
    default_credit,
    balance,
    housing,
    loan,
    contact,
    day_of_week,
    month,
    duration,
    campaign,
    pdays,
    previous,
    poutcome,
    y
)
SELECT
    NULLIF(TRIM(age_raw), '')::SMALLINT,
    NULLIF(TRIM(job_raw), ''),
    NULLIF(TRIM(marital_raw), ''),
    NULLIF(TRIM(education_raw), ''),
    TRIM(default_raw) = 'yes',
    NULLIF(TRIM(balance_raw), '')::INTEGER,
    TRIM(housing_raw) = 'yes',
    TRIM(loan_raw) = 'yes',
    NULLIF(TRIM(contact_raw), ''),
    NULLIF(TRIM(day_of_week_raw), '')::SMALLINT,
    NULLIF(TRIM(month_raw), ''),
    NULLIF(TRIM(duration_raw), '')::INTEGER,
    NULLIF(TRIM(campaign_raw), '')::SMALLINT,
    NULLIF(TRIM(pdays_raw), '')::INTEGER,
    NULLIF(TRIM(previous_raw), '')::SMALLINT,
    NULLIF(TRIM(poutcome_raw), ''),
    TRIM(y_raw) = 'yes'
FROM hw3_bank_staging;
"""

with engine.begin() as conn:
    result = conn.exec_driver_sql(insert_clean_sql)
    print("Rows inserted:", result.rowcount)


Rows inserted: 45211


In [12]:
with engine.connect() as conn:
    display(pd.read_sql("""
        SELECT
            (SELECT COUNT(*) FROM hw3_bank_staging) AS staging_rows,
            (SELECT COUNT(*) FROM hw3_bank_clean) AS clean_rows
    """, conn))

    display(pd.read_sql("""
        SELECT *
        FROM hw3_bank_clean
        ORDER BY record_id
        LIMIT 5
    """, conn))

    display(pd.read_sql("""
        SELECT
            COUNT(*) FILTER (WHERE default_credit IS TRUE) AS default_yes,
            COUNT(*) FILTER (WHERE housing IS TRUE) AS housing_yes,
            COUNT(*) FILTER (WHERE loan IS TRUE) AS loan_yes,
            COUNT(*) FILTER (WHERE y IS TRUE) AS subscribed_yes
        FROM hw3_bank_clean
    """, conn))

,staging_rows,clean_rows
0,45211,45211


,record_id,age,job,marital,education,default_credit,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,1,58,management,married,tertiary,False,2143,True,False,None,5,may,261,1,-1,0,None,False
1,2,44,technician,single,secondary,False,29,True,False,None,5,may,151,1,-1,0,None,False
2,3,33,entrepreneur,married,secondary,False,2,True,True,None,5,may,76,1,-1,0,None,False
3,4,47,blue-collar,married,None,False,1506,True,False,None,5,may,92,1,-1,0,None,False
4,5,33,None,single,None,False,1,False,False,None,5,may,198,1,-1,0,None,False


,default_yes,housing_yes,loan_yes,subscribed_yes
0,815,25130,7244,5289


In [13]:
with engine.connect() as conn:
    display(pd.read_sql("""
        SELECT 'staging' AS layer, COUNT(*) AS n_rows
        FROM hw3_bank_staging

        UNION ALL

        SELECT 'clean' AS layer, COUNT(*) AS n_rows
        FROM hw3_bank_clean
    """, conn))

,layer,n_rows
0,staging,45211
1,clean,45211


In [14]:
with engine.connect() as conn:
    display(pd.read_sql("""
        SELECT COUNT(*) AS total_rows
        FROM hw3_bank_clean
    """, conn))

    display(pd.read_sql("""
        SELECT COUNT(*) AS distinct_records
        FROM (
            SELECT DISTINCT
                age, job, marital, education,
                default_credit, balance, housing, loan,
                contact, day_of_week, month, duration,
                campaign, pdays, previous, poutcome, y
            FROM hw3_bank_clean
        ) AS unique_rows
    """, conn))


,total_rows
0,45211


,distinct_records
0,45211


In [15]:
with engine.connect() as conn:
    display(pd.read_sql("""
        SELECT
            COUNT(*) AS total_rows,

            COUNT(*) FILTER (WHERE job IS NULL) AS job_nulls,
            ROUND(
                COUNT(*) FILTER (WHERE job IS NULL) * 100.0
                / NULLIF(COUNT(*), 0), 2
            ) AS job_null_pct,

            COUNT(*) FILTER (WHERE education IS NULL) AS education_nulls,
            ROUND(
                COUNT(*) FILTER (WHERE education IS NULL) * 100.0
                / NULLIF(COUNT(*), 0), 2
            ) AS education_null_pct,

            COUNT(*) FILTER (WHERE contact IS NULL) AS contact_nulls,
            ROUND(
                COUNT(*) FILTER (WHERE contact IS NULL) * 100.0
                / NULLIF(COUNT(*), 0), 2
            ) AS contact_null_pct,

            COUNT(*) FILTER (WHERE poutcome IS NULL) AS poutcome_nulls,
            ROUND(
                COUNT(*) FILTER (WHERE poutcome IS NULL) * 100.0
                / NULLIF(COUNT(*), 0), 2
            ) AS poutcome_null_pct,

            COUNT(*) FILTER (WHERE marital IS NULL) AS marital_nulls,
            ROUND(
                COUNT(*) FILTER (WHERE marital IS NULL) * 100.0
                / NULLIF(COUNT(*), 0), 2
            ) AS marital_null_pct

        FROM hw3_bank_clean
    """, conn))

,total_rows,job_nulls,job_null_pct,education_nulls,education_null_pct,contact_nulls,contact_null_pct,poutcome_nulls,poutcome_null_pct,marital_nulls,marital_null_pct
0,45211,288,0.64,1857,4.11,13020,28.8,36959,81.75,0,0.0


In [16]:
with engine.connect() as conn:
    print("Special category values:")
    display(pd.read_sql("""
        SELECT 'job' AS column_name, job AS value, COUNT(*) AS n_rows
        FROM hw3_bank_clean
        WHERE LOWER(TRIM(job)) = 'unknown'
           OR TRIM(job) = ''
        GROUP BY job

        UNION ALL

        SELECT 'education', education, COUNT(*)
        FROM hw3_bank_clean
        WHERE LOWER(TRIM(education)) = 'unknown'
           OR TRIM(education) = ''
        GROUP BY education

        UNION ALL

        SELECT 'contact', contact, COUNT(*)
        FROM hw3_bank_clean
        WHERE LOWER(TRIM(contact)) = 'unknown'
           OR TRIM(contact) = ''
        GROUP BY contact

        UNION ALL

        SELECT 'poutcome', poutcome, COUNT(*)
        FROM hw3_bank_clean
        WHERE LOWER(TRIM(poutcome)) = 'unknown'
           OR TRIM(poutcome) = ''
        GROUP BY poutcome
    """, conn))

    print("\nNumeric domain checks:")
    display(pd.read_sql("""
        SELECT
            COUNT(*) FILTER (WHERE age <= 0) AS invalid_age,
            COUNT(*) FILTER (
                WHERE day_of_week NOT BETWEEN 1 AND 31
            ) AS invalid_day,
            COUNT(*) FILTER (WHERE duration < 0) AS negative_duration,
            COUNT(*) FILTER (WHERE campaign <= 0) AS invalid_campaign,
            COUNT(*) FILTER (
                WHERE pdays <> -1 AND pdays < 0
            ) AS invalid_pdays,
            COUNT(*) FILTER (WHERE previous < 0) AS negative_previous
        FROM hw3_bank_clean
    """, conn))

Special category values:


,column_name,value,n_rows



Numeric domain checks:


,invalid_age,invalid_day,negative_duration,invalid_campaign,invalid_pdays,negative_previous
0,0,0,0,0,0,0


An examination of categorical fields revealed no text values ​​of "unknown" or empty strings. However, SQL NULL values—identified during the initial audit—remain in the dataset. Checks of numerical fields showed no values ​​violating the established range rules. The value `pdays = -1` has been retained as a valid special code indicating the absence of prior contact.

In [17]:
query_1 = """
SELECT
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_1, conn))

,total_clients,subscribed_clients,subscription_rate_pct
0,45211,5289,11.7


Out of 45,211 clients, 5,289 opened a term deposit, which amounts to approximately 11.70%. This indicates a relatively low conversion rate for the campaign and may be an important factor when building a subscription prediction model.

In [18]:
query_2 = """
SELECT
    month,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY month
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_2, conn))

,month,total_clients,subscribed_clients,subscription_rate_pct
0,mar,477,248,51.99
1,dec,214,100,46.73
2,sep,579,269,46.46
3,oct,738,323,43.77
4,apr,2932,577,19.68
5,feb,2649,441,16.65
6,aug,6247,688,11.01
7,jun,5341,546,10.22
8,nov,3970,403,10.15
9,jan,1403,142,10.12


The highest share of term deposit subscriptions is observed in March (51.99%), December (46.73%), September (46.46%), and October (43.77%). The lowest conversion rates were recorded in May (6.72%), July (9.09%), and January (10.12%).

At the same time, the number of contacts varies significantly from month to month. For instance, there were 13,766 clients in May, compared to only 477 in March. Therefore, a high subscription percentage in a given month does not necessarily translate to the highest number of acquired clients.

In [19]:
query_3 = """
SELECT
    COALESCE(contact, 'unknown') AS contact_type,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY COALESCE(contact, 'unknown')
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_3, conn))

,contact_type,total_clients,subscribed_clients,subscription_rate_pct
0,cellular,29285,4369,14.92
1,telephone,2906,390,13.42
2,unknown,13020,530,4.07


The highest subscription rate is observed among clients contacted via cellular (mobile) connection — 14.92%.
In the dataset, 29,285 clients had a contact type of "cellular," 2,906 had "telephone," and for 13,020 records, the contact type is missing. The results indicate a relationship between the contact type and the subscription rate, but they do not establish a causal link.

In [20]:
query_4 = """
SELECT
    housing AS has_housing_loan,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY housing
ORDER BY housing DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_4, conn))

,has_housing_loan,total_clients,subscribed_clients,subscription_rate_pct
0,True,25130,1935,7.7
1,False,20081,3354,16.7


Clients without a housing loan have a higher rate of deposit subscriptions—16.70% compared to 7.70% among clients with a loan. This indicates a relationship between having a housing loan and the campaign outcome, but it does not prove causality.

In [21]:
query_5 = """
SELECT
    y AS subscribed,
    COUNT(*) AS total_clients,
    ROUND(AVG(balance), 2) AS avg_balance,
    MIN(balance) AS min_balance,
    MAX(balance) AS max_balance
FROM hw3_bank_clean
GROUP BY y
ORDER BY y DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_5, conn))

,subscribed,total_clients,avg_balance,min_balance,max_balance
0,True,5289,1804.27,-3058,81204
1,False,39922,1303.71,-8019,102127


Clients who signed up for a deposit had a higher average balance—1,804.27 versus 1,303.71 for those who did not. Both groups include clients with a negative balance.

In [22]:
query_6 = """
SELECT
    CASE
        WHEN age < 30 THEN 'Under 30'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY age_group
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_6, conn))

,age_group,total_clients,subscribed_clients,subscription_rate_pct
0,60+,1784,600,33.63
1,Under 30,5273,928,17.60
2,30-39,18089,1913,10.58
3,50-59,8410,785,9.33
4,40-49,11655,1063,9.12


The highest share of subscriptions is found among clients aged 60+ (33.63%), followed by clients under 30 (17.60%). The lowest conversion rate is observed in the 40–49 age group (9.12%).

In [23]:
query_7 = """
SELECT
    CASE
        WHEN campaign = 1 THEN '1 contact'
        WHEN campaign BETWEEN 2 AND 3 THEN '2-3 contacts'
        WHEN campaign BETWEEN 4 AND 5 THEN '4-5 contacts'
        ELSE '6+ contacts'
    END AS contact_group,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY contact_group
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_7, conn))

,contact_group,total_clients,subscribed_clients,subscription_rate_pct
0,1 contact,17544,2561,14.60
1,2-3 contacts,18026,2019,11.20
2,4-5 contacts,5286,456,8.63
3,6+ contacts,4355,253,5.81


The subscription rate decreases as the number of contacts increases: from 14.60% after one contact to 5.81% after six or more. This may indicate a decline in the effectiveness of repeated contacts.

In [24]:
query_8 = """
SELECT
    COALESCE(poutcome, 'unknown') AS previous_outcome,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY COALESCE(poutcome, 'unknown')
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_8, conn))

,previous_outcome,total_clients,subscribed_clients,subscription_rate_pct
0,success,1511,978,64.73
1,other,1840,307,16.68
2,failure,4901,618,12.61
3,unknown,36959,3386,9.16


The highest share of subscriptions is observed among clients who had a successful outcome in the previous campaign—64.73%. For clients with an unknown outcome from the previous campaign, the figure is only 9.16%.

In [25]:
query_9 = """
SELECT
    default_credit,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY default_credit
ORDER BY default_credit;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_9, conn))

,default_credit,total_clients,subscribed_clients,subscription_rate_pct
0,False,44396,5237,11.80
1,True,815,52,6.38


Among clients with no credit default, the share of subscriptions is 11.80%, whereas among clients with a default, it is 6.38%. Thus, clients without a credit default were more likely to subscribe to a deposit.

In [26]:
query_10 = """
SELECT
    CASE
        WHEN duration < 60 THEN 'Under 1 min'
        WHEN duration < 180 THEN '1-3 min'
        WHEN duration < 300 THEN '3-5 min'
        ELSE '5+ min'
    END AS call_duration_group,
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
    ROUND(
        COUNT(*) FILTER (WHERE y = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY call_duration_group
ORDER BY subscription_rate_pct DESC;
"""

with engine.connect() as conn:
    display(pd.read_sql(query_10, conn))

,call_duration_group,total_clients,subscribed_clients,subscription_rate_pct
0,5+ min,12329,3467,28.12
1,3-5 min,10345,1121,10.84
2,1-3 min,17878,692,3.87
3,Under 1 min,4659,9,0.19


The highest share of subscriptions is observed among clients with a call duration exceeding 5 minutes — 28.12%. For calls lasting up to 1 minute, the conversion rate is only 0.19%. At the same time, call duration could indicate target leakage if the model is intended to predict a subscription before the call begins.

## Reflection

In this assignment, I worked with the UCI Bank Marketing dataset containing 45,211 records. I loaded the data into PostgreSQL using a staging table and transformed it into a typed table. I checked missing values, duplicates, and numeric ranges, then used ten SQL queries to analyze subscription rates across different customer groups.

The results showed that subscription rates vary by age, contact type, and previous campaign outcomes. However, these relationships do not prove causation. Call duration may also cause target leakage if the model predicts subscription before a call begins.

For future improvements, I would add automated data quality checks and monitor data distributions and model performance.